# QphoX Optical Control Box - OpticalControlSystem

This layout of the optical components in the control box is shown in the following figure. 

![Drive Setup](./Optical_Control_Box_Architecture.png)


From left to right the setup includes four major sections with components within each of them. The relevant QCoDeS parameters which are associated with each section are also highlighted in italics. 

- SOURCE:
    - An internal laser
    - A port for an external laser
    - A 2x2 switch used to set the laser input

- ATTENUATOR:
    - A manual Variable Optical Attenuator (VOA)
    - A power meter to monitor optical power after the VOA

- MODULATOR:
    - A polarization controller (PC) 
    - An Electro-Optic Modulator (EOM) with an input for an RF signal
    - A power meter to monitor optical power after the EOM

- OUTPUT:
    - An Acousto-Optic Modulator (AOM) with its driver
    - A powermeter for measuring the optical power after the AOM
    - An 8x1 switch for setting the output channel. Six in use, two auxilliary

The rest of this notebook contains a minimum working example and the generic usage of the other parameters of the control box.

#### IMPORTANT
Ensure that the notebook is run using the conda virtual environment - "qphoxvenv".

--------------------------------------------------------------------------------------------------------------------------

## Import modules

The modules required for operation are OpticalControlSystem and qcodes

In [ ]:
from OpticalControlSystem import *
from qcodes import find_or_create_instrument

--------------------------------------------------------------------------------------------------------------------------

## Initialize the box

Initialize the optical control box with its IP address.
Auto tune-up the box to maximize its polarization and to bias the EOM to its midpoint.
Request a power at the output within the maximum permissible output power limit.

In [ ]:
qx_ocs = find_or_create_instrument(instrument_class=OpticalControlSystem, name="QX_OCS_SQC_Delivery", address="192.168.1.101")

--------------------------------------------------------------------------------------------------------------------------

## Automatic tune-up

Perform automatic tune-up of the control box. This will optimize the light polarization and EOM bias voltage for maximum RF signal generation at a given optical power. 

In [ ]:
qx_ocs.auto_tune_up(maximize_polarization=True, eom_setpoint='midpoint', eom_resolution=0.25, eom_plot=True)

--------------------------------------------------------------------------------------------------------------------------

## Setting the output channel

Set the output switch to the desired channel to transmit light to the corresponding photodiode. One can select between channels 1 through 6 to transmit to a photodiode channel. Setting the output switch to channel 0 disables the light.

In [ ]:
output_channel = 6
qx_ocs.output_switch.set(output_channel)

--------------------------------------------------------------------------------------------------------------------------

## Print control box status

Print a readable snapshot of the control box parameters.

In [ ]:
qx_ocs.print_readable_snapshot()

--------------------------------------------------------------------------------------------------------------------------

## Set output power

Set the power at the output of the box. Attach an optical power meter to the acitve output channel to measure the actual power emitted from the box. 

In [ ]:
required_output_power = 100e-9 # W

qx_ocs.set_output_power(required_output_power=required_output_power)

print(f"OpticalControlSystem '{qx_ocs.name}'")
print(f"Required output power: {required_output_power:.2e} W")
print(f"Measured attenuator power: {qx_ocs.attenuator_power.get():.2e} W")
print(f"Measured modulator power: {qx_ocs.modulator_power.get():.2e} W")
print(f"Measured output power: {qx_ocs.output_power.get():.2e} W")

The user can also specify two parameters manually to set the output power:

- Power correction factor:
    - This factor signifies the correction applied to the output power owing to changes in insertion loss or line resistances.
    - To estimate the correction factor, first assign a value of 1.0 and record the output power measured using an optical power meter.
    - Compute the ratio of the required output power and the measured output power as the power correction factor.

- Laser attenuation:
    - Estimate the attenuation required on the source if a low output power is required. Refer to Table 3 of the Application Note.

In [ ]:
# Optional

required_output_power = 10e-6 # W
power_correction_factor = 1.0 # Output channel loss factor
laser_attenuation = 0 # dB

qx_ocs.set_output_power(
    required_output_power=required_output_power,
    laser_attenuation=laser_attenuation,
    power_correction_factor=power_correction_factor)

print(f"OpticalControlSystem '{qx_ocs.name}'")
print(f"Laser attenuation: {qx_ocs.laser_attenuation.get()} dB")
print(f"Required output power: {required_output_power:.2e} W")
print(f"Measured attenuator power: {qx_ocs.attenuator_power.get():.2e} W")
print(f"Measured modulator power: {qx_ocs.modulator_power.get():.2e} W")
print(f"Measured output power: {qx_ocs.output_power.get():.2e} W")

--------------------------------------------------------------------------------------------------------------------------

## Output power stability

Stabilize the output power using a software-based PID loop that runs in a separate thread. The user can still access other functionalities of the optical control box such as:
- Changing the output switch channel
- Printing the readable snapshot of the control box

After stopping the power stability thread, obtain the accumulated power readings recorded during power stabilisation algorithm. For further analysis, the results can also be plotted and saved to the current working directory.

The PID is tuned to its default control parameters. However, they can be modified by specifying them as inputs to the function described below.

To repeat the process:
- Set a required output power
- Start the output power stabilisation
- Stop the output power stabilisation before setting a new output power

In [ ]:
# Default PID control parameters
k_p=0.24
k_i=0.05
k_d=0.12
averaging_time=0.2
sample_time=1.5

# Spawns a thread in the background to stabilise power
qx_ocs.start_output_power_stabilisation(k_p=k_p, k_i=k_i, k_d=k_d, averaging_time=averaging_time, sample_time=sample_time)

import time
time.sleep(0.25*60*60)
result = qx_ocs.stop_output_power_stabilisation()

Plot the power readings as a time series and as a probablity distribution by specifying the required power as a string.

In [ ]:
# Modify the file save prefix for power to better suit your application. For example:
power_prefix = "QX_OCS_TEST"
qx_ocs.plot_power_specs(result_dict=result, power_prefix=power_prefix)

--------------------------------------------------------------------------------------------------------------------------

## Shut down the instrument

Disable the light and turn off the instrument. Terminate the logging thread and save it to the current working directory.

In [ ]:
qx_ocs.close()

--------------------------------------------------------------------------------------------------------------------------

## Additional Controls

The following cells describe user actions accepted on changing the IP address and in each of the four sections - Source, Attenuator, Modulator, and Switch.

### IP Address

Change the IP address by providing a string of the for 192.168.1.xxx where xxx is in the range 1 to 127.
NOTE: Turn off and turn on the instrument for the change to take place.

In [ ]:
IP_address = '192.168.1.99'
qx_ocs.IP.set(IP_address)

--------------------------------------------------------------------------------------------------------------------------

### Source Control

The source functionality in the optical control is described through the following steps:
- Turn off the laser at the source
- Change the source to either an internal or an external laser depending on the current source of power
- Turn the laser back on

In [ ]:
# Turn off laser output
qx_ocs.laser_output('off')

# Determine current laser source and switch to the alternate source of power
if qx_ocs.laser_switch.get() == 'INTERNAL':
    qx_ocs.laser_switch.set('EXTERNAL')
else:
    qx_ocs.laser_switch.set('INTERNAL')

# Turn on laser output
qx_ocs.laser_output('on') 


--------------------------------------------------------------------------------------------------------------------------

### Attenuator Control

The power emitted from the laser source can be changed by setting its attenuation (in dB) of the VOA. 

In [ ]:
laser_attenuation = 0 # dB
qx_ocs.laser_attenuation.set(laser_attenuation) # dB
print(f"OpticalControlSystem '{qx_ocs.name}' | Laser set attenuation: {laser_attenuation} dB | \
 Laser measured voltage: {qx_ocs.laser_attenuation.get()} dB | \
 Measured attenuator power: {qx_ocs.attenuator_power.get():.2e} W")

--------------------------------------------------------------------------------------------------------------------------

### Modulator Control

The modulation of the light is controlled by the auto_tune_up function with the following options:
- Polarization controller
    - Provide a boolean input to maximize the polarization controllers
- EOM biasing
    - The EOM can be biased to one of the following three setpoints:
        - Minpoint
        - Midpoint
        - Maxpoint
    - The EOM can be sampled at a different resolution set by the user to change the precision of biasing the EOM at a given setpoint.

In [ ]:
maximize_polarization = False # or TRUE
eom_setpoint = 'midpoint' # or minpoint or maxpoint
eom_resolution = 0.25
eom_plot = True # or False

qx_ocs.auto_tune_up(maximize_polarization=maximize_polarization, eom_setpoint=eom_setpoint, eom_resolution=eom_resolution, eom_plot=eom_plot)

print(f"OpticalControlSystem '{qx_ocs.name}' | Measured output power: {qx_ocs.pm1_optical_power.get():.2e} W")

--------------------------------------------------------------------------------------------------------------------------

### Output Control

Change the output channel from which light is emitted. The expected inputs are from 1 to 8, where number 7 and 8 are the auxiliary ports. Setting 0 will disable light being emitted from the output switch.

In [ ]:
output_channel = 0 # 1-8 for light to be emitted from the output switch
qx_ocs.output_switch.set(output_channel)

--------------------------------------------------------------------------------------------------------------------------